# 04 — Nested Functions for Decorators

This is the 4th notebook in `03_Decorators`.

The purpose of this notebook is to build the exact foundation needed for decorators:

```text
Nested Functions → Closures → Wrapper Functions → Function Transformation
```

**Do not introduce `@decorator` syntax yet.** That belongs in the next notebook, `05_Basic_Decorators.ipynb`.

## Notebook Structure

```text
1. Introduction
2. What Are Nested Functions?
3. Defining a Function Inside Another Function
4. Calling the Inner Function
5. Outer and Inner Function Scope
6. Returning an Inner Function
7. Inner Functions Remember Enclosing Variables
8. Closures
9. Inspecting a Closure
10. Using Closures to Store State
11. Functions as Arguments + Nested Functions
12. Building a Wrapper Function
13. Wrapping Another Function
14. Adding Behavior Before and After a Function
15. Passing Arguments Through a Wrapper
16. Returning the Wrapper
17. Manual Function Decoration
18. Practical Examples
19. Common Mistakes
20. Connection to Decorators
21. Summary
```

# 1. Introduction

Decorators are built from several concepts already learned:

```text
Functions as Objects
        ↓
Higher-Order Functions
        ↓
Functions Returning Functions
        ↓
Nested Functions
        ↓
Closures
        ↓
Wrapper Functions
        ↓
Decorators
```

This notebook makes that progression explicit.

# 2. What Are Nested Functions?

A nested function is simply a function defined **inside another function**.

In [ ]:
def outer():
    def inner():
        print("Hello from inner")

    inner()

outer()

Here:

- `outer()` is the outer function.
- `inner()` is the nested/inner function.
- `inner()` exists inside the scope of `outer()`.

# 3. Defining a Function Inside Another Function

Start with a basic example.

In [ ]:
def outer():
    print("Outer function")

    def inner():
        print("Inner function")

    inner()

outer()

The inner function is normally not available as a standalone name outside `outer()`.

In [ ]:
def outer():
    def inner():
        print("Hello")

    inner()

outer()

Trying to use `inner()` outside the scope where it was defined results in a `NameError` because `inner` is a local name inside `outer`.

In [ ]:
def outer():
    def inner():
        print("Hello")

    inner()

outer()

try:
    inner()
except NameError as error:
    print(type(error).__name__)

# 4. Calling the Inner Function

Distinguish **defining** a function from **calling** it.

In [ ]:
def outer():
    def inner():
        print("Hello")

    print("Inner function has been defined")
    inner()

outer()

This creates the function:

```python
def inner():
```

This calls it:

```python
inner()
```

# 5. Outer and Inner Function Scope

An inner function can access variables from its enclosing function.

In [ ]:
def outer():
    message = "Hello"

    def inner():
        print(message)

    inner()

outer()

In [ ]:
def outer():
    name = "Python"

    def inner():
        print("Learning", name)

    inner()

outer()

This ability to access an enclosing variable leads naturally into closures.

# 6. Returning an Inner Function

Returning an inner function is extremely important for decorators.

In [ ]:
def outer():
    def inner():
        print("Hello")

    return inner

result = outer()

print(result)
result()

Remember the distinction:

```python
return inner
```

returns the function.

Whereas:

```python
return inner()
```

calls the function and returns its result.

In [ ]:
def outer():
    def inner():
        return "Hello"

    return inner

function = outer()

print(function())

# 7. Inner Functions Remember Enclosing Variables

Consider:

In [ ]:
def outer():
    message = "Hello from outer"

    def inner():
        print(message)

    return inner

function = outer()
function()

Even though `outer()` has already finished executing, `inner()` can still access `message`.

This is the key behavior behind a **closure**.

# 8. Closures

A closure is:

> A function that remembers and retains access to variables from its enclosing scope, even after the enclosing function has finished executing.

In [ ]:
def create_greeting(name):
    def greet():
        print("Hello", name)

    return greet

greeting = create_greeting("Komal")
greeting()

In [ ]:
def create_greeting(name):
    def greet():
        print("Hello", name)

    return greet

greeting1 = create_greeting("Alice")
greeting2 = create_greeting("Bob")

greeting1()
greeting2()

Each returned function retains access to the value that was supplied when it was created.

```text
create_greeting("Alice")
        ↓
    creates greet()
        ↓
    remembers "Alice"
```

# 9. Inspecting a Closure

Python provides `__closure__` for inspecting closure information.

This is useful for understanding closures, although you normally do not manipulate `__closure__` directly.

In [ ]:
def create_greeting(name):
    def greet():
        print("Hello", name)

    return greet

greeting = create_greeting("Alice")

print(greeting.__closure__)

In [ ]:
print(greeting.__closure__[0].cell_contents)

The closure cell contains:

```text
Alice
```

# 10. Using Closures to Store State

Closures can retain state between calls.

In [ ]:
def counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

count = counter()

print(count())
print(count())
print(count())

Output:

```text
1
2
3
```

`nonlocal count` allows the nested function to modify the variable belonging to the enclosing function.

This connects to the earlier Functions and Scope / Nested Functions and Closures material, but here the focus is on why the mechanism is useful when building wrappers.

# 11. Functions as Arguments + Nested Functions

Now combine the previous concepts.

In [ ]:
def execute(func):
    def wrapper():
        print("Before function")
        func()
        print("After function")

    wrapper()

def greet():
    print("Hello")

execute(greet)

Output:

```text
Before function
Hello
After function
```

The pieces are:

```text
execute()
   │
   ├── receives a function
   │
   └── defines wrapper()
          │
          ├── extra behavior
          ├── original function
          └── extra behavior
```

# 12. Building a Wrapper Function

A **wrapper** is a function that surrounds another function and adds behavior.

In [ ]:
def wrapper(func):
    def inner():
        print("Starting")
        func()
        print("Finished")

    return inner

def greet():
    print("Hello")

new_greet = wrapper(greet)
new_greet()

Output:

```text
Starting
Hello
Finished
```

This is the most important example in the notebook.

# 13. Wrapping Another Function

Show the transformation explicitly.

In [ ]:
def wrapper(func):
    def inner():
        print("Starting")
        func()
        print("Finished")

    return inner

def greet():
    print("Hello")

print("Original:")
greet()

In [ ]:
greet = wrapper(greet)

print("Wrapped:")
greet()

The transformation is:

```text
Original greet
     ↓
wrapper(greet)
     ↓
new wrapper function
     ↓
greet now refers to wrapper
```

This is **manual decoration**.

# 14. Adding Behavior Before and After a Function

A wrapper can add behavior around the original function.

In [ ]:
def add_logging(func):
    def wrapper():
        print("Function is starting")
        func()
        print("Function is finished")

    return wrapper

def greet():
    print("Hello")

greet = add_logging(greet)
greet()

In [ ]:
def show_message():
    print("Python is powerful")

show_message = add_logging(show_message)
show_message()

The important concept is:

> The original function's behavior is preserved, while additional behavior is added around it.

# 15. Passing Arguments Through a Wrapper

This is essential before learning real decorators.

First, use a wrapper designed for one specific parameter.

In [ ]:
def wrapper(func):
    def inner(name):
        print("Before")
        func(name)
        print("After")

    return inner

def greet(name):
    print("Hello", name)

greet = wrapper(greet)
greet("Komal")

Output:

```text
Before
Hello Komal
After
```

Now use a wrapper for two parameters.

In [ ]:
def wrapper(func):
    def inner(a, b):
        print("Before")
        result = func(a, b)
        print("After")
        return result

    return inner

def add(a, b):
    return a + b

add = wrapper(add)

print(add(10, 20))

Output:

```text
Before
After
30
```

The wrapper needs to accommodate the original function's parameters.

This naturally prepares the reader for:

```python
*args
**kwargs
```

which will become important in the decorator notebooks that follow.

# 16. Returning the Wrapper

Focus on the core pattern.

In [ ]:
def decorator(func):
    def wrapper():
        func()

    return wrapper

def greet():
    print("Hello")

greet = decorator(greet)
greet()

Break it down:

```text
decorator(func)
       ↓
defines wrapper()
       ↓
returns wrapper
```

After:

```python
greet = decorator(greet)
```

the name `greet` refers to the returned wrapper.

# 17. Manual Function Decoration

Introduce the word **decoration**, but still avoid `@` syntax.

In [ ]:
def add_extra_behavior(func):
    def wrapper():
        print("Before")
        func()
        print("After")

    return wrapper

def greet():
    print("Hello")

greet = add_extra_behavior(greet)
greet()

A decorator is fundamentally a callable that receives a function and returns a modified or replacement function.

For now, save the actual syntax:

```python
@add_extra_behavior
```

for the next notebook.

# 18. Practical Examples

## Example 1 — Logging

In [ ]:
def log_function(func):
    def wrapper():
        print("Calling function...")
        func()
        print("Function completed.")

    return wrapper

def greet():
    print("Hello!")

greet = log_function(greet)
greet()

## Example 2 — Authentication Concept

In [ ]:
def check_access(func):
    def wrapper():
        logged_in = True

        if logged_in:
            func()
        else:
            print("Access denied")

    return wrapper

def dashboard():
    print("Welcome to dashboard")

dashboard = check_access(dashboard)
dashboard()

This demonstrates a common decorator use case without introducing formal decorator syntax.

## Example 3 — Timing Concept

Keep this foundational rather than implementing actual timing logic.

In [ ]:
def measure_function(func):
    def wrapper():
        print("Start timing")
        func()
        print("Stop timing")

    return wrapper

def process():
    print("Processing...")

process = measure_function(process)
process()

Later, a decorator can replace these messages with actual timing logic.

## Example 4 — Validation

In [ ]:
def validate(func):
    def wrapper():
        print("Validating...")
        func()

    return wrapper

def process_data():
    print("Processing data")

process_data = validate(process_data)
process_data()

# 19. Common Mistakes

## Mistake 1 — Calling instead of returning

Wrong:

```python
return inner()
```

Correct:

```python
return inner
```

## Mistake 2 — Forgetting to return the wrapper

Wrong:

```python
def decorator(func):
    def wrapper():
        func()
```

Correct:

```python
def decorator(func):
    def wrapper():
        func()

    return wrapper
```

## Mistake 3 — Calling the original function too early

Wrong:

```python
def decorator(func):
    func()

    def wrapper():
        print("Wrapper")

    return wrapper
```

`func()` executes immediately when the decorator function runs.

## Mistake 4 — Losing the return value

Potential problem:

```python
def wrapper(func):
    def inner():
        func()

    return inner
```

If the original function returns a value, the wrapper above does not return it.

Correct pattern:

```python
def wrapper(func):
    def inner(a, b):
        result = func(a, b)
        return result

    return inner
```

## Mistake 5 — Wrapper does not accept the required arguments

Suppose:

```python
def greet(name):
    print("Hello", name)
```

A wrapper like:

```python
def inner():
    func()
```

cannot correctly wrap `greet(name)`.

This leads directly to `*args` and `**kwargs` in the following decorator material.

# 20. Connection to Decorators

The complete conceptual chain is:

```text
Function
   ↓
Function passed as argument
   ↓
Nested function
   ↓
Nested function calls original function
   ↓
Nested function adds behavior
   ↓
Nested function is returned
   ↓
Original function is replaced by returned wrapper
   ↓
Manual decorator
   ↓
@decorator syntax
```

The key pattern is:

In [ ]:
def decorator(func):
    def wrapper():
        # extra behavior
        func()
        # extra behavior

    return wrapper

At this point, you should understand **what a decorator does**, even though you have not learned the `@` syntax yet.

# 21. Summary

## Nested functions

```python
def outer():
    def inner():
        pass
```

## Returning functions

```python
return inner
```

## Closures

```python
def outer(value):
    def inner():
        print(value)

    return inner
```

## Wrappers

```python
def decorator(func):
    def wrapper():
        func()

    return wrapper
```

## Manual decoration

```python
function = decorator(function)
```

## Core idea

> A decorator works by receiving a function, creating a wrapper around it, and returning that wrapper.

### What this notebook does **not** cover deeply

To keep the sequence clean, do not go deeply into:

- `@decorator` syntax
- `*args` / `**kwargs` in decorators
- `functools.wraps`
- decorators with arguments
- class decorators
- advanced decorator patterns

Those belong to the following notebooks.

**Next:** `05_Basic_Decorators.ipynb` — where we finally introduce the `@decorator` syntax and build real basic decorators.